# Task B judge study: label the Gemini 3.8 Flash segments with candidate judges, exactly as the package does

Task B annotators label the **947 segments** of the 36 Task A windows (Gemini 3.8 Flash segmentation) with one
action each. To choose the labeling judge fairly, every candidate judge must label those segments **exactly as the
real pipeline would**. This notebook does **no segmenting or labeling of its own**: it only runs package commands
(**reasoning-anatomy v1.8.9**) and keeps the books.

**Why there is a recovery step.** None of the six Task A Gemini 3.8 Flash runs has a complete segmentation
(`stages/04_atomicity/quality.json`: `all_boundaries_resolved = false`; 34 to 6,123 unresolved tokens). With its
default settings the package then refuses to label (`relabel`: *exit 3, Segmentation remains unresolved … Use the
recovery command to derive a new run*), and where a segmentation request of the source run failed, `relabel` stops
(*exit 6*) because no saved response exists for it. So the notebook first does what the package prescribes:

1. **Recovery (section 4, once, segmentation model unchanged = Gemini 3.8 Flash):** the package's own recovery command
   `anatomy resegment --source-run <run> --run-dir <new run>` (the `recovery_command` it prints). Every saved
   segmentation response is replayed; only requests without a saved response are sent again. The command is repeated
   on its own result until a round sends **no new request**: then every segmentation request is answered from saved
   responses, which is exactly what `relabel` needs. If a run still has unresolved boundaries after that (the
   segmenter's own answer, e.g. *no additional safe boundary*, is replayed and cannot change), one last derived run
   sets the package's own switch `require_complete_segmentation = false` (no request is sent, segments identical,
   verified). The package then labels every resolved segment and marks the segments inside unresolved boundaries
   `TECHNICAL_MISSING` (its own rule), for every judge alike. Turn this off with `ALLOW_UNRESOLVED = False` to keep
   the default (those models are then not labeled at all).
2. **Segmentation check (free):** the recovered segments are compared with the website's segments; a website segment
   whose package segments changed, or that lies in an unresolved boundary, is **excluded for every judge** and listed.
3. **Judges (section 6):** for each judge and creator model, `anatomy check-judge`, then
   `anatomy relabel --source-run <recovered run>` (*segmentation is replayed, never re-run*; same prompt, request
   rendering, earlier-context selection, referenced-setup context, COMMIT and answer-section rules, schema validation,
   retries and receipts as the real pipeline), `segments_identical_to_source` checked with the package's own
   function, then `anatomy export-annotations`, and the labels are mapped onto the website's segments.

**Everything is read from and written to Drive** (`Reasoning Behavior/TaskB/`).

**Run order:** sections 1–3 in every session; section 4 (recovery, once: re-running it never pays twice); section 5;
one cell of section 6 per judge (re-running a cell resumes and never pays twice); section 7.

## 1. Settings

In [ ]:
RB          = "/content/drive/MyDrive/Reasoning Behavior"     # project folder on Drive
PACKAGE_ZIP = f"{RB}/reasoning-anatomy-v1.8.9.zip"
PACKAGE_VERSION = "1.8.9"
TASKA       = f"{RB}/TaskA"
TASKB       = f"{RB}/TaskB"
KEY         = f"{TASKA}/private/key.csv"                       # window -> trace (private; never published)
OUTPUTS     = f"{RB}/Outputs"                                  # traces.jsonl.gz of every creator model
SOURCE      = f"{TASKA}/judge_study/g3.8-flash"                 # the Gemini 3.8 Flash runs = the segmentation of record
TRACES_JS   = f"{TASKB}/platform/traces.js"                     # the windows and segments annotators label
SITE_TRACES = "https://amirhosseinyari2002.github.io/trace-task-b/traces.js"   # used once to fill TRACES_JS
WORK        = f"{TASKB}/judge_study"                            # one package run per judge x model, costs.csv
JUDGES_OUT  = f"{TASKB}/judges"                                 # <judge>.json for the researcher page
RECOVERED   = f"{WORK}/recovered"                               # the recovered Gemini 3.8 Flash runs (section 4)
LABELED     = f"{WORK}/labels"                                  # one relabel run per judge x model
MAX_RECOVERY_ROUNDS = 4        # `anatomy resegment` rounds per model until a round sends no new request
ALLOW_UNRESOLVED    = True     # package switch require_complete_segmentation=false on the recovered run (see above)

import os, sys, json, re, csv, glob, gzip, hashlib, datetime, subprocess, bisect
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if not IN_COLAB:                                                 # local test with a mirrored folder
    RB = os.environ.get("TASKB_RB", RB)
    TASKA, TASKB, OUTPUTS = f"{RB}/TaskA", f"{RB}/TaskB", f"{RB}/Outputs"
    KEY, SOURCE = f"{TASKA}/private/key.csv", f"{TASKA}/judge_study/g3.8-flash"
    TRACES_JS, WORK, JUDGES_OUT = f"{TASKB}/platform/traces.js", f"{TASKB}/judge_study", f"{TASKB}/judges"
    RECOVERED, LABELED = f"{WORK}/recovered", f"{WORK}/labels"

## 2. Setup (once per Colab session): Drive, the package, API keys

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    init = Path("/content/reasoning-anatomy/src/reasoning_anatomy/__init__.py")
    if not (init.exists() and f'__version__ = "{PACKAGE_VERSION}"' in init.read_text()):
        subprocess.run(["rm", "-rf", "/content/reasoning-anatomy"], check=True)
        subprocess.run(["unzip", "-q", "-o", PACKAGE_ZIP, "-d", "/content/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", "/content/reasoning-anatomy[research,api]"], check=True)
    from google.colab import userdata
    for k in ("GEMINI_API_KEY", "ANTHROPIC_API_KEY"):          # key icon -> add the secret
        try: os.environ[k] = userdata.get(k)
        except Exception: print(f"no {k} secret: judges of that provider stop at their check until you add it")
    if "/content/reasoning-anatomy/src" not in sys.path: sys.path.insert(0, "/content/reasoning-anatomy/src")
installed = subprocess.run([sys.executable, "-c", "import reasoning_anatomy as r; print(r.__version__)"], capture_output=True, text=True).stdout.strip()
print("reasoning-anatomy:", installed or "not installed")
if IN_COLAB: assert installed == PACKAGE_VERSION, f"reasoning-anatomy {PACKAGE_VERSION} is needed (found {installed or 'none'})"
for d in (f"{TASKB}/platform", WORK, RECOVERED, LABELED, JUDGES_OUT, f"{TASKB}/annotations", f"{TASKB}/results"): os.makedirs(d, exist_ok=True)

## 3. Inputs: the windows and segments (Drive), the key, the traces, the Gemini 3.8 Flash runs

`TaskB/platform/traces.js` is the exact file the annotators' website uses (copied once from the site, then always
read from Drive). Its text and segment hashes are checked, and every window's text is checked against the trace
it was cut from.

In [ ]:
def read_traces_js(path):
    raw = open(path, encoding="utf-8").read()
    k = raw.index("window.TRACES = ") + len("window.TRACES = ")
    src = json.loads(re.search(r"window\.TASKB_SOURCE = (\{.*?\});", raw).group(1))
    return json.loads(raw[k: raw.rindex(";")]), src
if not os.path.exists(TRACES_JS):
    import urllib.request
    open(TRACES_JS, "w", encoding="utf-8").write(urllib.request.urlopen(SITE_TRACES, timeout=60).read().decode("utf-8"))
    print("copied the website's traces.js to", TRACES_JS)
WINDOWS, SOURCE_INFO = read_traces_js(TRACES_JS)
WIN = {w["id"]: w for w in WINDOWS}
for w in WINDOWS:
    assert hashlib.sha256(w["text"].encode("utf-8")).hexdigest() == w["sha256"], w["id"] + ": text hash"
    assert hashlib.sha256(json.dumps(w["segs"]).encode()).hexdigest() == w["seg_sha256"], w["id"] + ": segmentation hash"
N_SEG = sum(len(w["segs"]) for w in WINDOWS)
print(f"{len(WINDOWS)} windows, {N_SEG} segments; segmentation {SOURCE_INFO['judge']}")

KEYROWS = {r["item_id"]: r for r in csv.DictReader(open(KEY, encoding="utf-8")) if str(r["practice"]).lower() != "true"}
missing = sorted(set(WIN) - set(KEYROWS))
assert not missing, f"windows not in key.csv: {missing}"
MODELS = sorted({KEYROWS[i]["model_code"] for i in WIN})

def load_traces(codes):
    """code -> trace_id, completion_text, reasoning (the text the windows were cut from), as the Task A kit reads them."""
    want, found = set(codes), {}
    for path in sorted(glob.glob(os.path.join(OUTPUTS, "*", "traces.jsonl.gz"))):
        prefixes = {c.split("-")[0] for c in want - set(found)}
        if not prefixes: break
        with gzip.open(path, "rt", encoding="utf-8") as f:
            first = True
            for line in f:
                row = json.loads(line)
                if first:
                    first = False
                    if row["code"].split("-")[0] not in prefixes: break
                if row["code"] in want: found[row["code"]] = {k: row[k] for k in ("trace_id", "completion_text", "reasoning")}
    return found
TRACE = load_traces([KEYROWS[i]["code"] for i in WIN])
for i, w in WIN.items():
    k, t = KEYROWS[i], TRACE[KEYROWS[i]["code"]]
    comp, reas = t["completion_text"], t["reasoning"]
    assert comp.count(reas) == 1, i + ": reasoning is not a unique part of completion_text"
    base = comp.index(reas) + int(k["char_start"])
    assert comp[base: base + len(w["text"])] == w["text"], i + ": the website window text differs from the trace"
    w["_base"], w["_trace_id"], w["_model"] = base, t["trace_id"], k["model_code"]
print("every window matches its trace exactly; models:", MODELS)
for m in MODELS:
    assert os.path.exists(f"{SOURCE}/{m}/resolved_config.json"), f"no Gemini 3.8 Flash run for {m} in {SOURCE}"

## 4. Package helpers, recovery of the Gemini 3.8 Flash segmentation (once), and the segmentation check

`recover(model)` runs the package's recovery command round by round (each round derives a new run from the previous
one; nothing is ever changed in place) and writes `TaskB/judge_study/recovered/<MODEL>/record.json` naming the run every
judge will relabel. Its only cost is the re-sent Gemini 3.8 Flash segmentation requests (the package's own accounting,
written to `costs.csv`). Re-running the cell resumes and never pays twice.

In [ ]:
def anatomy(*args):
    """Run the package's command line; returns (exit code, short message, parsed JSON result)."""
    print("  $ anatomy " + " ".join(map(str, args)), flush=True)
    p = subprocess.run([sys.executable, "-m", "reasoning_anatomy", *map(str, args)], capture_output=True, text=True)
    lines = (p.stdout or "").strip().splitlines()
    last = lines[-1] if lines else (p.stderr or "").strip()[-400:]
    try: parsed = json.loads(last)
    except ValueError: parsed = None
    short = (parsed.get("error") or parsed.get("status")) if isinstance(parsed, dict) else last
    return p.returncode, str(short)[:400], parsed

_INFO = r"""
import json, sys
from pathlib import Path
from reasoning_anatomy.ledger import RunLedger
rd = Path(sys.argv[1]); L = RunLedger(rd, read_only=True)
try:
    out = {"stages": {s: (L.stage_info(s) or {}).get("state") for s in ("04_atomicity", "05_semantics", "07_report")},
           "segmentation_usage": L.metadata("segmentation_actual_usage")}
finally:
    L.close()
q = rd / "stages/04_atomicity/quality.json"
q = json.loads(q.read_text()) if q.exists() else {}
out["quality"] = {k: q.get(k) for k in ("all_boundaries_resolved", "unresolved_tokens", "source_tokens", "blocking_issue_counts", "root_issue_counts")}
out["require_complete_segmentation"] = json.loads((rd / "resolved_config.json").read_text())["segmentation"].get("require_complete_segmentation", True)
print(json.dumps(out))
"""
def run_info(rd):
    """Stage states, segmentation usage, quality and the completeness switch of a package run (read only)."""
    p = subprocess.run([sys.executable, "-c", _INFO, str(rd)], capture_output=True, text=True)
    if p.returncode: raise SystemExit(f"cannot read the run {rd}: {p.stderr[-400:]}")
    return json.loads(p.stdout)
def same_segments(a, b):
    """The package's own check (the one `relabel` reports as segments_identical_to_source)."""
    p = subprocess.run([sys.executable, "-c", "import sys; from pathlib import Path; from reasoning_anatomy.cli import _same_segments; "
                        "print(_same_segments(Path(sys.argv[1]), Path(sys.argv[2])))", str(a), str(b)], capture_output=True, text=True)
    if p.returncode: raise SystemExit(f"segment comparison failed: {p.stderr[-400:]}")
    return p.stdout.strip() == "True"

def append_cost(row):
    """One row per judge x model x stage (replaced when recorded again: usage is cumulative)."""
    path = Path(WORK) / "costs.csv"
    rows = list(csv.DictReader(open(path))) if path.exists() else []
    if row["stage"] != "check-judge":
        rows = [r for r in rows if not (r["judge"] == row["judge"] and r["model_code"] == row["model_code"] and r["stage"] == row["stage"])]
    rows.append({k: str(v) for k, v in row.items()})
    fields = list(dict.fromkeys(list(row) + [k for r in rows for k in r]))
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, restval=""); w.writeheader(); w.writerows(rows)
    os.replace(tmp, path)
now = lambda: datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")

def recover(m):
    rec = Path(RECOVERED) / m; rec.mkdir(parents=True, exist_ok=True)
    record = rec / "record.json"
    if record.exists(): return json.loads(record.read_text())
    prev, rounds, spent, calls_total = Path(SOURCE) / m, [], 0.0, 0
    for r in range(1, MAX_RECOVERY_ROUNDS + 1):
        rd = rec / f"round{r}"
        code, short, res = anatomy("resegment", "--source-run", prev, "--run-dir", rd)
        info = run_info(rd) if (rd / "run.sqlite3").exists() else None
        if code not in (0, 3) or not info or info["stages"]["04_atomicity"] != "SEALED":
            print(f"  {m}: recovery round {r} not finished (exit {code}): {short}\n  run the cell again to resume."); return None
        u = info["segmentation_usage"] or {}
        calls = int(u.get("calls") or 0); spent += float(u.get("cost_usd") or 0); calls_total += calls
        q = info["quality"]
        rounds.append({"run": str(rd), "new_requests": calls, "cost_usd": u.get("cost_usd"), "cached_requests": u.get("cached_requests"),
                       "all_boundaries_resolved": q["all_boundaries_resolved"], "unresolved_tokens": q["unresolved_tokens"],
                       "blocking_issue_counts": q["blocking_issue_counts"]})
        print(f"  {m} round {r}: {calls} new segmentation request(s), {u.get('cached_requests')} replayed; "
              f"unresolved tokens {q['unresolved_tokens']}; issues {q['blocking_issue_counts']}")
        if calls == 0: break                              # fixed point: every request is answered from saved responses
        prev = rd
    else:
        print(f"  {m}: still sending new requests after {MAX_RECOVERY_ROUNDS} rounds (a request keeps failing). "
              f"Run the cell again for more rounds after raising MAX_RECOVERY_ROUNDS, or leave {m} out.")
        return None
    final, complete = rd, rounds[-1]["all_boundaries_resolved"]
    if not complete:
        if not ALLOW_UNRESOLVED:
            out = {"model": m, "run": None, "rounds": rounds, "complete": False, "labelable": False}
            record.write_text(json.dumps(out, indent=1)); print(f"  {m}: incomplete and ALLOW_UNRESOLVED = False: not labeled"); return out
        cfg = rec / "segmentation-config.json"; cfg.write_text(json.dumps({"require_complete_segmentation": False}))
        rd = rec / "labelable"
        code, short, res = anatomy("resegment", "--source-run", final, "--run-dir", rd, "--segmentation-config", cfg)
        info = run_info(rd) if (rd / "run.sqlite3").exists() else None
        if code not in (0, 3) or not info or info["stages"]["04_atomicity"] != "SEALED":
            print(f"  {m}: switch run not finished (exit {code}): {short}"); return None
        u = info["segmentation_usage"] or {}
        if int(u.get("calls") or 0) != 0 or not same_segments(final, rd) or info["require_complete_segmentation"] is not False:
            raise SystemExit(f"{m}: the switch run sent requests or changed segments; stop and report this")
        rounds.append({"run": str(rd), "new_requests": 0, "switch": {"require_complete_segmentation": False}})
        final = rd
    out = {"model": m, "run": str(final), "rounds": rounds, "complete": complete, "labelable": True,
           "unresolved_tokens": rounds[-1].get("unresolved_tokens", rounds[-2]["unresolved_tokens"] if len(rounds) > 1 else None),
           "segments_identical_to_source_run": same_segments(Path(SOURCE) / m, final),
           "segmentation_new_requests": calls_total, "segmentation_cost_usd": round(spent, 6), "created": now()}
    if out["unresolved_tokens"] is None: out["unresolved_tokens"] = rounds[0]["unresolved_tokens"]
    append_cost({"time": now(), "judge": "segmentation-recovery", "model_id": "gemini-3.8-flash", "model_code": m, "stage": "04_recovery",
                 "requests": calls_total, "input_tokens": "", "output_tokens": "", "thinking_tokens": "", "cache_read_tokens": "", "cache_write_tokens": "",
                 "price_in": "", "price_out": "", "cost_usd": round(spent, 6), "run_dir": str(final)})
    record.write_text(json.dumps(out, indent=1))
    return out

RECORD = {}
for m in MODELS:
    print(f"\n[recovery {m}]")
    r = recover(m)
    if r: RECORD[m] = r
print()
for m in MODELS:
    r = RECORD.get(m)
    print(f"{m}: " + ("NOT READY (run this cell again)" if not r else
          f"labelable={r['labelable']} complete={r['complete']} unresolved_tokens={r.get('unresolved_tokens')} "
          f"segments identical to Task A run={r.get('segments_identical_to_source_run')} new requests={r.get('segmentation_new_requests')} "
          f"cost ${r.get('segmentation_cost_usd')}"))

### Segmentation check (free): recovered segments vs the website's segments

The website shows the Gemini 3.8 Flash boundaries inside each window (snapped to word starts, as in Task A's Part B).
A package segment belongs to the website segment that contains its first non-space character; the segment that begins
before the window (in the grey context) belongs to the window's first segment. First the Task A runs are mapped (every
website segment must be covered: this proves the runs are the segmentation of record), then the recovered runs. A
website segment is **excluded for every judge** when its package segments changed in the recovery
(`segmentation_changed`) or lie in an unresolved boundary (`unresolved_boundary`, labeled `TECHNICAL_MISSING` by the
package). The list is saved to `TaskB/judge_study/recovered/excluded.json`.

In [ ]:
def export_of(run_dir, out):
    if not os.path.exists(out):
        code, short, _ = anatomy("export-annotations", "--run-dir", run_dir, "--output", out)
        if code != 0 or not os.path.exists(out): raise SystemExit(f"export of {run_dir} failed: {short}")
    return out

def map_export(export_path, items, strict=True):
    """website segments of `items` -> list of the package segments that make them up (dicts from the export)."""
    doc = json.load(open(export_path, encoding="utf-8"))
    if doc.get("format") != "saae-human-alignment-v1" or doc.get("kind") != "prediction":
        raise SystemExit(f"{export_path}: not an `anatomy export-annotations` prediction file")
    by_tid = {t["trace_id"]: t for t in doc["traces"]}
    out = {}
    for i in items:
        w = WIN[i]; t = by_tid.get(w["_trace_id"])
        if t is None: raise SystemExit(f"{export_path}: trace of window {i} is missing")
        comp = TRACE[KEYROWS[i]["code"]]["completion_text"]
        if t["text_sha256"] != hashlib.sha256(comp.encode("utf-8")).hexdigest():
            raise SystemExit(f"{i}: the run's trace text differs from Outputs")
        base, end = w["_base"], w["_base"] + len(w["text"])
        starts = [base + a for _, a, _ in w["segs"]]
        parts = [[] for _ in w["segs"]]
        for s in t["segments"]:
            if s["end"] <= base or s["start"] >= end: continue                 # outside the window
            txt = comp[s["start"]: s["end"]]
            core = s["start"] + (len(txt) - len(txt.lstrip()))
            if not txt.strip() or core >= end: continue                         # whitespace only, or starts after it
            k = bisect.bisect_right(starts, max(core, base)) - 1
            parts[k].append(s)
        empty = [k for k, p in enumerate(parts) if not p]
        if empty and strict: raise SystemExit(f"{i}: website segment(s) {empty} contain no package segment: the runs are not the segmentation of record")
        out[i] = parts
    return out

LABELS = ("CONSTRUCT", "INSPECT", "REVISE", "REDIRECT", "NONACTION")
bounds = lambda parts: [(s["start"], s["end"]) for s in parts]
EXCLUDED, one = {}, 0
for m in MODELS:
    items = [i for i in WIN if WIN[i]["_model"] == m]
    src = map_export(export_of(f"{SOURCE}/{m}", f"{RECOVERED}/{m}/source-export.json"), items)
    for p in src.values(): one += sum(len(x) == 1 for x in p)
    r = RECORD.get(m)
    if not r or not r["labelable"]:
        for i in items: EXCLUDED[i] = {str(k): "model_not_labelable" for k in range(len(WIN[i]["segs"]))}
        continue
    rec = map_export(export_of(r["run"], f"{RECOVERED}/{m}/recovered-export.json"), items, strict=False)
    for i in items:
        for k, (a, b) in enumerate(zip(src[i], rec[i])):
            why = ("segmentation_changed" if bounds(a) != bounds(b) else
                   "unresolved_boundary" if any(not s.get("boundary_resolved", True) for s in b) else None)
            if why: EXCLUDED.setdefault(i, {})[str(k)] = why
print(f"Task A runs map onto all {N_SEG} website segments ({one} one-to-one).")
from collections import Counter
cnt = Counter(v for d in EXCLUDED.values() for v in d.values())
print(f"excluded for every judge: {sum(cnt.values())} of {N_SEG} website segments {dict(cnt)}")
for i in sorted(EXCLUDED, key=lambda x: [w["id"] for w in WINDOWS].index(x)):
    print(f"  {WIN[i]['label']} ({i}): segments {sorted(int(k) + 1 for k in EXCLUDED[i])} -> {sorted(set(EXCLUDED[i].values()))}")
json.dump({"excluded": EXCLUDED, "counts": dict(cnt), "records": RECORD, "created": now()}, open(f"{RECOVERED}/excluded.json", "w"), indent=1)

## 5. Candidate judges (edit here) and the package calls

Same format as the Task A judge study (`settings` = the package's AnnotationConfig fields; `price` = USD per 1M
tokens, output includes thinking). Each judge's settings and prices are written **once** to
`TaskB/judge_study/<judge>/judge-config.json`; a later edit of a judge that has already run is refused (give it a
new name instead), so a resumed run is never mixed. Check IDs and prices on your pricing pages first.

In [ ]:
JUDGES = {
  "g3.8-flash":             {"provider": "gemini", "model": "gemini-3.8-flash", "price": [0.75, 3.75],
                             "settings": {"temperature": None, "gemini_thinking_level": "low", "gemini_thinking_budget": None, "max_output_tokens": 4096},
                             "note": "the segmentation judge; introductory price to 2026-12-31, then 1.50 / 7.50"},
  "g3.5-flash-lite":        {"provider": "gemini", "model": "gemini-3.5-flash-lite", "price": [0.30, 2.50],
                             "settings": {"temperature": None, "gemini_thinking_level": "minimal", "gemini_thinking_budget": None}},
  "g3.5-flash-lite-think":  {"provider": "gemini", "model": "gemini-3.5-flash-lite", "price": [0.30, 2.50],
                             "settings": {"temperature": None, "gemini_thinking_level": "low", "gemini_thinking_budget": None, "max_output_tokens": 4096}},
  "g3.1-flash-lite":        {"provider": "gemini", "model": "gemini-3.1-flash-lite", "price": [0.25, 1.50],
                             "settings": {"temperature": None, "gemini_thinking_level": "minimal", "gemini_thinking_budget": None}},
  "g2.5-flash-lite":        {"provider": "gemini", "model": "gemini-2.5-flash-lite", "price": [0.10, 0.40],
                             "settings": {"temperature": None, "gemini_thinking_level": None, "gemini_thinking_budget": 0}},
  "g4-hydrogen":            {"provider": "gemini", "model": "gemini-4-hydrogen", "price": [1.50, 7.50],
                             "settings": {"temperature": None, "gemini_thinking_level": "low", "gemini_thinking_budget": None, "max_output_tokens": 4096}},
  "claude-haiku-5.5":       {"provider": "claude", "model": "claude-haiku-5-5", "price": [0.10, 0.50],
                             "settings": {"temperature": None, "gemini_thinking_level": None, "gemini_thinking_budget": None, "claude_thinking": "disabled", "reasoning_effort": "low"}},
  "claude-haiku-5.5-think": {"provider": "claude", "model": "claude-haiku-5-5", "price": [0.10, 0.50],
                             "settings": {"temperature": None, "gemini_thinking_level": None, "gemini_thinking_budget": None, "claude_thinking": "adaptive", "reasoning_effort": "low", "max_output_tokens": 8192}},
}

def receipts_usage(run_dir, stages=("05_semantics",)):
    """Exact usage from the package's per-request receipts (as the Task A kit computes it)."""
    out = {}
    for stage in stages:
        files = glob.glob(str(Path(run_dir) / "stages" / stage / "requests" / "*.receipt.json"))
        n = pin = pout = think = cread = cwrite = 0
        for f in files:
            try: d = json.load(open(f))
            except (ValueError, OSError): continue
            for r in d.get("transport_attempts") or ([d["response"]] if "response" in d else []):
                meta = r.get("response_metadata") or {}
                p = r.get("prompt_tokens") or meta.get("prompt_token_count") or 0
                o = r.get("output_tokens") or meta.get("candidates_token_count") or 0
                t = meta.get("thoughts_token_count") or 0
                total = meta.get("total_token_count")
                usage = meta.get("usage") or {}
                cread += int(usage.get("cache_read_input_tokens") or 0); cwrite += int(usage.get("cache_creation_input_tokens") or 0)
                n += 1; pin += p; pout += (total - p) if total else (o + t); think += t
        out[stage] = {"requests": n, "input_tokens": pin, "output_tokens": pout, "thinking_tokens": think,
                      "cache_read_tokens": cread, "cache_write_tokens": cwrite}
    return out
def cost(u, price):
    read, write = u.get("cache_read_tokens", 0), u.get("cache_write_tokens", 0)
    billed_in = u["input_tokens"] - read - write + 0.1 * read + 1.25 * write
    return round((billed_in * price[0] + u["output_tokens"] * price[1]) / 1e6, 6)
print("judges:", list(JUDGES))

## 6. Run the judges: one cell per judge

Each cell: `check-judge` (a few cents; a wrong model ID, setting or key stops here and creates nothing), then for each
creator model `anatomy relabel --source-run <recovered run>` and `anatomy export-annotations`, the cost from the
receipts, and finally `TaskB/judges/<judge>.json`. The package labels **every segment of the 36 traces** (not only
the window segments), as the real pipeline would; only the window segments are compared with the annotators.
When the recovered run has unresolved boundaries, `relabel` finishes all stages and then reports exit 3
(`segmentation_incomplete`): that is the package's report of the unresolved boundaries, not a failure; the cell
then checks that labeling is sealed and runs the package's own segment-identity check.
Re-run a cell to resume after an interruption: the package resumes its own run and never pays twice.
Runs from the earlier version of this notebook (`TaskB/judge_study/<judge>/<MODEL>`) are not used and can be deleted.

In [ ]:
def website_labels(parts, excluded=None):
    """one label per website segment; None when not comparable."""
    if excluded: return None, excluded
    acts = [s.get("action") for s in parts]
    if len(set(acts)) == 1 and acts[0] in LABELS: return acts[0], "ok"
    if any(a is None for a in acts): return None, "missing"
    if "COMMIT" in acts: return None, "commit"
    return None, "parts_disagree"

def run_judge(name, models=None):
    J = JUDGES[name]; price = J["price"]; jdir = Path(LABELED) / name; jdir.mkdir(parents=True, exist_ok=True)
    overrides = dict(J["settings"], input_usd_per_million=price[0], output_usd_per_million=price[1],
                     pricing_snapshot=f"TaskB judge study ({name}); verify on the provider's pricing page")
    cfg = jdir / "judge-config.json"
    if cfg.exists():
        if json.loads(cfg.read_text()) != overrides:
            raise SystemExit(f"{name} already ran with other settings ({cfg}); give the new settings a new judge name")
    else:
        cfg.write_text(json.dumps(overrides, indent=1))
    todo = [m for m in (models or MODELS) if RECORD.get(m, {}).get("labelable")]
    skipped = [m for m in (models or MODELS) if m not in todo]
    if skipped: print("not labelable / not recovered yet (section 4):", skipped)
    check = jdir / "check-judge.json"
    if not check.exists() or not json.loads(check.read_text()).get("passed"):
        code, short, report = anatomy("check-judge", "--judge-model", J["model"], "--judge-config", cfg)
        if report: check.write_text(json.dumps(report, indent=1))
        billed = [t for t in (report or {}).get("tasks", {}).values() if t.get("ok")]
        u = {"requests": len(billed), "input_tokens": sum(t.get("input_tokens") or 0 for t in billed),
             "output_tokens": sum(t.get("output_tokens") or 0 for t in billed), "thinking_tokens": 0, "cache_read_tokens": 0, "cache_write_tokens": 0}
        append_cost({"time": now(), "judge": name, "model_id": J["model"], "model_code": "-", "stage": "check-judge", **u,
                     "price_in": price[0], "price_out": price[1], "cost_usd": cost(u, price), "run_dir": ""})
        if code != 0:
            raise SystemExit(f"judge check failed: {short} (details in {check}); nothing was created")
    results = {}
    for m in todo:
        rd, src = jdir / m, Path(RECORD[m]["run"])
        print(f"\n[{name} / {m}]", flush=True)
        code, short, res = anatomy("relabel", "--source-run", src, "--run-dir", rd, "--annotator", J["provider"],
                                   "--annotator-model", J["model"], "--annotation-config", cfg)
        info = run_info(rd) if (rd / "run.sqlite3").exists() else None
        done = bool(info) and info["stages"]["05_semantics"] == "SEALED" and info["stages"]["07_report"] == "SEALED"
        if code == 0 and isinstance(res, dict) and res.get("status") == "completed":
            identical = res.get("segments_identical_to_source") is True
        elif code == 3 and done and not RECORD[m]["complete"] and info["require_complete_segmentation"] is False:
            identical = same_segments(src, rd)             # the package's check; relabel skips it when it reports exit 3
        else:
            print(f"  not finished (exit {code}): {short}\n  run this cell again to resume."); results[m] = f"not finished: {short}"; continue
        if not identical:
            raise SystemExit(f"{m}: the relabel run's segments differ from the recovered Gemini 3.8 Flash run")
        code, short, _ = anatomy("export-annotations", "--run-dir", rd, "--output", rd / "export.json")
        if code != 0: results[m] = f"export failed: {short}"; continue
        u = receipts_usage(rd)["05_semantics"]; c = cost(u, price)
        append_cost({"time": now(), "judge": name, "model_id": J["model"], "model_code": m, "stage": "05_semantics", **u,
                     "price_in": price[0], "price_out": price[1], "cost_usd": c, "run_dir": str(rd)})
        results[m] = f"labeled, ${c:.4f}"
    print(json.dumps(results, indent=1))
    if todo and all((jdir / m / "export.json").exists() for m in todo) and all(RECORD.get(m) for m in MODELS):
        return write_judge(name)
    print("judge file not written yet: some models are not finished (run section 4 / this cell again)")

def write_judge(name):
    J = JUDGES[name]; jdir = Path(LABELED) / name
    labels, actions, parts_n, reasons = {}, {}, {}, {}
    for m in MODELS:
        items = [i for i in WIN if WIN[i]["_model"] == m]
        if not RECORD[m]["labelable"]:
            for i in items:
                n = len(WIN[i]["segs"]); labels[i] = [None] * n; actions[i] = [[] for _ in range(n)]; parts_n[i] = [0] * n
                reasons["model_not_labelable"] = reasons.get("model_not_labelable", 0) + n
            continue
        mp = map_export(jdir / m / "export.json", items, strict=False)
        for i, parts in mp.items():
            res = [website_labels(p, EXCLUDED.get(i, {}).get(str(k))) for k, p in enumerate(parts)]
            labels[i] = [r[0] for r in res]
            actions[i] = [[s.get("action") for s in p] for p in parts]
            parts_n[i] = [len(p) for p in parts]
            for _, why in res: reasons[why] = reasons.get(why, 0) + 1
    costs = [r for r in csv.DictReader(open(Path(WORK) / "costs.csv")) if r["judge"] == name]
    first = next(m for m in MODELS if RECORD[m]["labelable"])
    run = json.load(open(jdir / first / "export.json"))["run"]
    out = {"format": "trace-task-b-judge/v2", "judge": name, "provider": J["provider"], "model": J["model"],
           "returned_model_version": run.get("returned_model_version"), "settings": J["settings"], "price": J["price"],
           "package_version": PACKAGE_VERSION,
           "protocol": "anatomy resegment (the package's recovery command, Gemini 3.8 Flash, until no new request; "
                       "require_complete_segmentation=false where boundaries stay unresolved) of TaskA/judge_study/g3.8-flash/<MODEL>; "
                       "anatomy relabel of the recovered run (segmentation replayed, identical), the package's own 05_semantics stage, "
                       "anatomy export-annotations; labels mapped onto the website segments; changed or unresolved segments excluded for every judge",
           "segmentation": SOURCE_INFO, "seg_sha256": {i: WIN[i]["seg_sha256"] for i in WIN},
           "segmentation_recovery": {m: {k: RECORD[m].get(k) for k in ("complete", "labelable", "unresolved_tokens", "segments_identical_to_source_run", "segmentation_new_requests")} for m in MODELS},
           "labels": labels, "package_actions": actions, "package_segments_per_website_segment": parts_n,
           "comparable": reasons, "cost_usd": round(sum(float(r["cost_usd"]) for r in costs), 4),
           "requests": sum(int(float(r["requests"] or 0)) for r in costs), "created": now()}
    path = Path(JUDGES_OUT) / f"{name}.json"
    json.dump(out, open(path, "w"), ensure_ascii=False)
    dist = {}
    for v in labels.values():
        for x in v: dist[x] = dist.get(x, 0) + 1
    print(f"wrote {path}: {reasons}; labels {dist}; cost so far ${out['cost_usd']}")
    return out

In [ ]:
run_judge("g3.8-flash")

In [ ]:
run_judge("g3.5-flash-lite")

In [ ]:
run_judge("claude-haiku-5.5")

In [ ]:
run_judge("claude-haiku-5.5-think")

Optional judges:

In [ ]:
# run_judge("g2.5-flash-lite")
# run_judge("g3.5-flash-lite-think")
# run_judge("g3.1-flash-lite")
# run_judge("g4-hydrogen")

## 7. Costs, label distributions, judge–judge agreement

The decision is made on the **researcher page**: Agreement → Judges → load `TaskB/judges/*.json` and
`TaskA/private/key.csv`. Download *all metrics (JSON)* there and save it in `TaskB/results/`.

In [ ]:
import pandas as pd
c = pd.read_csv(f"{WORK}/costs.csv")
display(c.groupby(["judge", "stage"])[["requests", "input_tokens", "output_tokens", "cost_usd"]].sum())
J = {p.stem: json.load(open(p)) for p in sorted(Path(JUDGES_OUT).glob("*.json"))}
rows = []
for n, d in J.items():
    flat = [x for v in d["labels"].values() for x in v]
    rows.append({"judge": n, "cost $": d["cost_usd"], "comparable": d["comparable"].get("ok", 0), **{l: flat.count(l) for l in LABELS}, "not comparable": flat.count(None)})
display(pd.DataFrame(rows).set_index("judge"))
def kappa(a, b):
    p = [(x, y) for x, y in zip(a, b) if x and y]
    if not p: return float("nan")
    po = sum(x == y for x, y in p) / len(p)
    pe = sum((sum(x == l for x, _ in p) / len(p)) * (sum(y == l for _, y in p) / len(p)) for l in LABELS)
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")
names = list(J); flat = {n: [x for w in WINDOWS for x in J[n]["labels"][w["id"]]] for n in names}
display(pd.DataFrame([[round(kappa(flat[a], flat[b]), 3) for b in names] for a in names], index=names, columns=names))